# Date exploration

In [13]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from datetime import datetime

from utils import extract_date_from_recording_id

In [25]:
data_path = "data/dataset.parquet"
df = pd.read_parquet(data_path)


In [27]:
df.head()

,recording_id,start_time,end_time,audspec_lengthL1norm_sma_iqr1_2,audspec_lengthL1norm_sma_iqr2_3,audspec_lengthL1norm_sma_iqr1_3,audspec_lengthL1norm_sma_percentile50_0,audspec_lengthL1norm_sma_stddev,audspec_lengthL1norm_sma_amean,audspecRasta_lengthL1norm_sma_iqr1_2,...,logHNR_sma_de_iqr2_3,logHNR_sma_de_iqr1_3,logHNR_sma_de_percentile50_0,logHNR_sma_de_stddev,F0final_sma_ff0_nnz,patient_short_id,label,age,sex,audio_quality
13946,patient_0000/2021-09-17,576,776,1.493601,0.472047,1.965648,2.107094,1.026020,1.629121,3.432586,...,2.196668,2.462965,0.370440,2.792117,0.923077,patient_0000,0,0.738636,1,0.761684
14070,patient_0000/2021-09-17,3008,3208,0.276523,0.181809,0.458331,0.774141,0.294968,0.704204,0.505588,...,0.160649,0.178956,0.016412,0.104311,1.000000,patient_0000,0,0.738636,1,0.913146
14071,patient_0000/2021-09-17,3076,3276,0.232940,0.161391,0.394331,0.761261,0.268565,0.692789,0.465115,...,0.204947,0.269043,0.074313,0.140533,1.000000,patient_0000,0,0.738636,1,0.921234
14072,patient_0000/2021-09-17,3076,3276,0.211107,0.261235,0.472341,0.866216,0.319627,0.833112,0.275575,...,0.196939,0.277636,0.082456,0.143403,1.000000,patient_0000,0,0.738636,1,0.921234
14073,patient_0000/2021-09-17,3076,3276,0.246978,0.158901,0.405880,0.779031,0.273051,0.702328,0.494150,...,0.206352,0.269329,0.072969,0.140931,1.000000,patient_0000,0,0.738636,1,0.921234


In [28]:
print(f"Shape of the dataset: {df.shape}")
unique_patients = df["patient_short_id"].unique()
print(f"Number of unique patients: {len(unique_patients)}")
print(f"Number of unique recording dates: {df['recording_id'].apply(extract_date_from_recording_id).nunique()}")
print("Columns in the dataset:")
for col in df.columns:
    print(f" - {col}")

Shape of the dataset: (73542, 789)
Number of unique patients: 14
Number of unique recording dates: 248
Columns in the dataset:
 - recording_id
 - start_time
 - end_time
 - audspec_lengthL1norm_sma_iqr1_2
 - audspec_lengthL1norm_sma_iqr2_3
 - audspec_lengthL1norm_sma_iqr1_3
 - audspec_lengthL1norm_sma_percentile50_0
 - audspec_lengthL1norm_sma_stddev
 - audspec_lengthL1norm_sma_amean
 - audspecRasta_lengthL1norm_sma_iqr1_2
 - audspecRasta_lengthL1norm_sma_iqr2_3
 - audspecRasta_lengthL1norm_sma_iqr1_3
 - audspecRasta_lengthL1norm_sma_percentile50_0
 - audspecRasta_lengthL1norm_sma_stddev
 - audspecRasta_lengthL1norm_sma_amean
 - pcm_RMSenergy_sma_iqr1_2
 - pcm_RMSenergy_sma_iqr2_3
 - pcm_RMSenergy_sma_iqr1_3
 - pcm_RMSenergy_sma_percentile50_0
 - pcm_RMSenergy_sma_stddev
 - pcm_RMSenergy_sma_amean
 - pcm_zcr_sma_iqr1_2
 - pcm_zcr_sma_iqr2_3
 - pcm_zcr_sma_iqr1_3
 - pcm_zcr_sma_percentile50_0
 - pcm_zcr_sma_stddev
 - pcm_zcr_sma_amean
 - audspec_lengthL1norm_sma_de_iqr1_2
 - audspec_leng

In [29]:
# Recordings per patient
recordings_per_patient = df.groupby("patient_short_id").size()
print("\nRecordings per patient:")
print(f"  Mean: {recordings_per_patient.mean():.1f}")
print(f"  Median: {recordings_per_patient.median():.1f}")
print(f"  Min: {recordings_per_patient.min()}")
print(f"  Max: {recordings_per_patient.max()}")


Recordings per patient:
  Mean: 5253.0
  Median: 4765.5
  Min: 1407
  Max: 13946


In [18]:
# Label distribution
print(f"\n{'=' * 70}")
print("LABEL DISTRIBUTION")
print("=" * 70)
label_counts = df["label"].value_counts()
print("Overall:")
print(f"  Stable (0): {label_counts[0]:,} ({label_counts[0] / len(df) * 100:.1f}%)")
print(
    f"  Pre-hospitalization (1): {label_counts[1]:,} ({label_counts[1] / len(df) * 100:.1f}%)"
)

print("\nPer patient:")
for patient in sorted(unique_patients):
    patient_df = df[df["patient_short_id"] == patient]
    patient_labels = patient_df["label"].value_counts()
    n_stable = patient_labels.get(0, 0)
    n_prehosp = patient_labels.get(1, 0)
    print(
        f"  {patient}: {n_stable} stable, {n_prehosp} pre-hosp "
        f"({n_prehosp / (n_stable + n_prehosp) * 100:.1f}% positive)"
    )


LABEL DISTRIBUTION
Overall:
  Stable (0): 62,614 (85.1%)
  Pre-hospitalization (1): 10,928 (14.9%)

Per patient:
  patient_0000: 6654 stable, 722 pre-hosp (9.8% positive)
  patient_0001: 3985 stable, 818 pre-hosp (17.0% positive)
  patient_0002: 13332 stable, 614 pre-hosp (4.4% positive)
  patient_0003: 2729 stable, 910 pre-hosp (25.0% positive)
  patient_0004: 5223 stable, 397 pre-hosp (7.1% positive)
  patient_0005: 5525 stable, 675 pre-hosp (10.9% positive)
  patient_0006: 2004 stable, 845 pre-hosp (29.7% positive)
  patient_0007: 1135 stable, 272 pre-hosp (19.3% positive)
  patient_0008: 1319 stable, 393 pre-hosp (23.0% positive)
  patient_0009: 2263 stable, 643 pre-hosp (22.1% positive)
  patient_0010: 3763 stable, 965 pre-hosp (20.4% positive)
  patient_0011: 1393 stable, 1309 pre-hosp (48.4% positive)
  patient_0012: 4993 stable, 1464 pre-hosp (22.7% positive)
  patient_0013: 8296 stable, 901 pre-hosp (9.8% positive)


In [30]:
# Temporal information
df["recording_date"] = extract_date_from_recording_id(df["recording_id"])

print("\nRecording date range:")
print(f"  First recording: {df['recording_date'].min().date()}")
print(f"  Last recording: {df['recording_date'].max().date()}")
print(
    f"  Total span: {(df['recording_date'].max() - df['recording_date'].min()).days} days"
)

print("\nFollow-up period per patient:")
for patient in sorted(unique_patients):
    patient_df = df[df["patient_short_id"] == patient]
    first_date = patient_df["recording_date"].min()
    last_date = patient_df["recording_date"].max()
    follow_up_days = (last_date - first_date).days
    n_recordings = len(patient_df)
    print(
        f"  {patient}: {follow_up_days} days ({first_date.date()} to {last_date.date()}), {n_recordings} recordings"
    )


Recording date range:
  First recording: 2021-09-13
  Last recording: 2022-08-21
  Total span: 342 days

Follow-up period per patient:
  patient_0000: 327 days (2021-09-17 to 2022-08-10), 7376 recordings
  patient_0001: 196 days (2021-09-21 to 2022-04-05), 4803 recordings
  patient_0002: 320 days (2021-10-01 to 2022-08-17), 13946 recordings
  patient_0003: 296 days (2021-09-17 to 2022-07-10), 3639 recordings
  patient_0004: 342 days (2021-09-13 to 2022-08-21), 5620 recordings
  patient_0005: 327 days (2021-09-26 to 2022-08-19), 6200 recordings
  patient_0006: 90 days (2021-12-17 to 2022-03-17), 2849 recordings
  patient_0007: 209 days (2021-10-30 to 2022-05-27), 1407 recordings
  patient_0008: 97 days (2021-12-29 to 2022-04-05), 1712 recordings
  patient_0009: 186 days (2022-02-07 to 2022-08-12), 2906 recordings
  patient_0010: 168 days (2022-01-29 to 2022-07-16), 4728 recordings
  patient_0011: 247 days (2021-11-04 to 2022-07-09), 2702 recordings
  patient_0012: 278 days (2021-11-10 

In [31]:
# Feature information
print(f"\n{'=' * 70}")
print("FEATURE INFORMATION")
print("=" * 70)

metadata_cols = [
    "recording_id",
    "patient_short_id",
    "label",
    "recording_date",  # Added by our extraction
]
feature_cols = [col for col in df.columns if col not in metadata_cols]

print(f"Number of acoustic features: {len(feature_cols)}")
print("\nFeature categories:")


FEATURE INFORMATION
Number of acoustic features: 786

Feature categories:


In [32]:
# Count feature types
feature_types = {}
for col in feature_cols:
    feature_type = col.split("_")[0]
    feature_types[feature_type] = feature_types.get(feature_type, 0) + 1

for ftype, count in sorted(feature_types.items()):
    print(f"  {ftype}: {count} features")

  F0final: 13 features
  age: 1 features
  audSpec: 312 features
  audio: 1 features
  audspec: 12 features
  audspecRasta: 12 features
  end: 1 features
  jitterDDP: 12 features
  jitterLocal: 12 features
  logHNR: 12 features
  mfcc: 168 features
  pcm: 204 features
  sex: 1 features
  shimmerLocal: 12 features
  start: 1 features
  voicingFinalUnclipped: 12 features


## Select top features

In [22]:
# Number of top features to select
X = df[feature_cols]  # Features dataframe
y = df["label"]  # Target labels

num_features = 300
print(f"\nSelecting top {num_features} features among {X.shape[-1]} based on ANOVA F-value")

# Feature selection using ANOVA F-value
score_func = f_classif

selector = SelectKBest(score_func=score_func, k=num_features)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support()]

# Dataframe with selected top features
df_selected_features = pd.DataFrame(X_selected, columns=selected_feature_names)


Selecting top 300 features among 786 based on ANOVA F-value


In [23]:
df_selected_features.head()

,start_time,end_time,audspec_lengthL1norm_sma_iqr1_2,audspec_lengthL1norm_sma_iqr2_3,audspec_lengthL1norm_sma_iqr1_3,audspec_lengthL1norm_sma_percentile50_0,audspec_lengthL1norm_sma_stddev,audspec_lengthL1norm_sma_amean,pcm_RMSenergy_sma_iqr1_2,pcm_RMSenergy_sma_iqr1_3,...,shimmerLocal_sma_de_iqr1_3,shimmerLocal_sma_de_percentile50_0,shimmerLocal_sma_de_stddev,logHNR_sma_de_amean,logHNR_sma_de_iqr1_2,logHNR_sma_de_iqr2_3,logHNR_sma_de_iqr1_3,logHNR_sma_de_percentile50_0,F0final_sma_ff0_nnz,age
0,576.0,776.0,1.493601,0.472047,1.965648,2.107094,1.026020,1.629121,0.158997,0.260397,...,0.001939,-0.002217,0.001780,1.931104,0.266297,2.196668,2.462965,0.370440,0.923077,0.738636
1,3008.0,3208.0,0.276523,0.181809,0.458331,0.774141,0.294968,0.704204,0.023495,0.042543,...,0.021546,-0.004579,0.011209,0.078137,0.018307,0.160649,0.178956,0.016412,1.000000,0.738636
2,3076.0,3276.0,0.232940,0.161391,0.394331,0.761261,0.268565,0.692789,0.033983,0.062686,...,0.015020,-0.006321,0.013358,0.140955,0.064097,0.204947,0.269043,0.074313,1.000000,0.738636
3,3076.0,3276.0,0.211107,0.261235,0.472341,0.866216,0.319627,0.833112,0.030678,0.071954,...,0.016372,-0.006988,0.013194,0.138365,0.080697,0.196939,0.277636,0.082456,1.000000,0.738636
4,3076.0,3276.0,0.246978,0.158901,0.405880,0.779031,0.273051,0.702328,0.033906,0.062770,...,0.014925,-0.006396,0.013354,0.140678,0.062978,0.206352,0.269329,0.072969,1.000000,0.738636


In [24]:
# Reconstruct the final dataframe with metadata and selected features
df_final = pd.concat([df[metadata_cols].reset_index(drop=True), df_selected_features], axis=1)
print(f"\nFinal dataframe shape with selected features: {df_final.shape}")

# Save the final dataframe to a new parquet file
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
final_data_path = f"data/dataset_{num_features}_selected_features_{current_time}.parquet"
df_final.to_parquet(final_data_path)
print(f"Final dataframe saved to {final_data_path}")


Final dataframe shape with selected features: (73542, 304)
Final dataframe saved to data/dataset_300_selected_features_20260120_125655.parquet
